In [ ]:
import sys
import os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)


In [ ]:
import torch
import random
import numpy as np
import csv

In [ ]:
from models.semcovert import create_network
from utils.video_utils import load_video_as_tensor,split_video_tensor, get_video_files

In [ ]:
def seet_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seet_seed(42)

In [ ]:
def load_videos(data_dir, target_size=(240,320), max_frames=5): 
    
    video_files = get_video_files(data_dir)
    if not video_files:
        raise ValueError(f"No video files found in {data_dir}")
    
    print(f"Found {len(video_files)} video files: {[f.name for f in video_files]}")
    
    all_video_batches = []
    for video_file in video_files:
        video_tensor = load_video_as_tensor(
            str(video_file), 
            target_size=target_size
        )
        video_tensor = split_video_tensor(video_tensor, max_frames)
        if video_tensor.numel() > 0:
            for i in range(video_tensor.shape[0]):
                all_video_batches.append(video_tensor[i])
            del video_tensor  
        else:
            print(f"  Failed to load video: {video_file}")
    
    if not all_video_batches:
        raise ValueError("No valid video data loaded")
    
    all_video_tensor = torch.stack(all_video_batches, dim=0)
    del all_video_batches

    indices = torch.randperm(all_video_tensor.shape[0])
    if len(indices) % 2 != 0:
        indices = indices[:-1] # Ensure even number of videos
        
    mid_count = len(indices) // 2
    cover_indices = indices[:mid_count]
    secret_indices = indices[mid_count:]

    cover_videos = all_video_tensor[cover_indices]
    secret_videos = all_video_tensor[secret_indices]
    del all_video_tensor  
    print(f"Loaded {cover_videos.shape[0]} cover videos and secret videos")
    
    return cover_videos, secret_videos

In [ ]:
def embedding(model, cover_video, secret_video, batch_size, device):
    cover_z = []
    stego_z = []

    with torch.no_grad():
        for i in range(0, cover_video.shape[0], batch_size):
            batch_cover = cover_video[i:i + batch_size].to(device)
            batch_secret = secret_video[i:i + batch_size].to(device) 
            
            output = model(batch_cover, batch_secret)
            cover_z_chunk = output['z_cover'].cpu().numpy()
            stego_z_chunk = output['z_fused'].cpu().numpy()
            cover_z.append(cover_z_chunk)
            stego_z.append(stego_z_chunk)

    cover_z = np.concatenate(cover_z, axis=0)
    stego_z = np.concatenate(stego_z, axis=0)

    return cover_z, stego_z


In [ ]:
def calculate_metrics(cover_z, stego_z):
    cover_flat = cover_z.reshape(cover_z.shape[0], -1)
    stego_flat = stego_z.reshape(stego_z.shape[0], -1)
    
    # MSE
    mse = np.mean((cover_flat - stego_flat) ** 2)
    
    # Cosine Similarity
    cos_sim = np.mean(np.sum(cover_flat * stego_flat, axis=1) / (
        np.linalg.norm(cover_flat, axis=1) * np.linalg.norm(stego_flat, axis=1) + 1e-8))
    
    # Wasserstein Distance
    from scipy.stats import wasserstein_distance
    w_dist = np.mean([
        wasserstein_distance(cover_flat[:, j], stego_flat[:, j])
        for j in range(cover_flat.shape[1])
    ])
    
    return mse, cos_sim, w_dist

In [ ]:
model_cfg = {
        'depth': 4,
        'dim': 96,
        'use_channel': True,
        'vae_config': {
            'dim': 96,
            'z_dim': 16,
            'dim_mult': [1, 2, 4, 4],
            'num_res_blocks': 2,
            'attn_scales': [],
            'temperal_downsample': [False, True, True],
            'dropout': 0.0
        },
        'channel_config': {
            'channel_type': 'AWGN',  # Channel type
            'snr': 25,               # Signal-to-noise ratio
        },    
    }

In [ ]:
model_path = "YOUR_MODEL_PATH"  # Path to your model weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_network(model_cfg)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

In [ ]:
csv_file = "YOUR_CSV_FILE_PATH"  # Path to save the CSV file
os.makedirs(os.path.dirname(csv_file), exist_ok=True)
with open(csv_file, 'w', newline='') as csvfile:
    header = ['data','mse', 'cosine_similarity', 'wasserstein_distance']
    writer = csv.writer(csvfile)
    writer.writerow(header)

In [ ]:
data_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for UCF101 Dataset
data_name = os.path.basename(data_dir)
batch_size = 16
max_batch = 1000
cover_videos, secret_videos = load_videos(data_dir, target_size=(240,320), max_frames=5)
if cover_videos.shape[0] > max_batch:
    cover_videos = cover_videos[:max_batch]
    secret_videos = secret_videos[:max_batch]
cover_z, stego_z = embedding(model, cover_videos, secret_videos, batch_size, device)
mse, cos_sim, w_dist = calculate_metrics(cover_z, stego_z)
del cover_z, stego_z
del cover_videos, secret_videos
with open(csv_file, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([data_name, mse, cos_sim, w_dist])

print(f"Results for {data_name}: MSE={mse}, Cosine Similarity={cos_sim}, Wasserstein Distance={w_dist}")

In [ ]:
data_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for DAVIS Dataset
data_name = os.path.basename(data_dir)
batch_size = 4
max_batch = 500
cover_videos, secret_videos = load_videos(data_dir, target_size=(480,640), max_frames=5)
if cover_videos.shape[0] > max_batch:
    cover_videos = cover_videos[:max_batch]
    secret_videos = secret_videos[:max_batch]
cover_z, stego_z = embedding(model, cover_videos, secret_videos, batch_size, device)
mse, cos_sim, w_dist = calculate_metrics(cover_z, stego_z)
del cover_z, stego_z
del cover_videos, secret_videos
with open(csv_file, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([data_name, mse, cos_sim, w_dist])

print(f"Results for {data_name}: MSE={mse}, Cosine Similarity={cos_sim}, Wasserstein Distance={w_dist}")

In [ ]:
data_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for MOT17 Dataset
data_name = os.path.basename(data_dir)
batch_size = 1
max_batch = 100
cover_videos, secret_videos = load_videos(data_dir, target_size=(1080,1920), max_frames=5)
if cover_videos.shape[0] > max_batch:
    cover_videos = cover_videos[:max_batch]
    secret_videos = secret_videos[:max_batch]
cover_z, stego_z = embedding(model, cover_videos, secret_videos, batch_size, device)
mse, cos_sim, w_dist = calculate_metrics(cover_z, stego_z)
del cover_z, stego_z
del cover_videos, secret_videos
with open(csv_file, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([data_name, mse, cos_sim, w_dist])

print(f"Results for {data_name}: MSE={mse}, Cosine Similarity={cos_sim}, Wasserstein Distance={w_dist}")